# Fed-ISIC2019 6-Client — Kaggle 확장 실험 B (μ 재확인 + 개선 설정 18라운드 + FedBN 18라운드, 에폭 기준)
`NOWEAZ/fedisic-federated-framework` 브랜치 `feature/v2-extended-experiments` 최신 커밋을 사용합니다. 기존 결과는 브랜치 `results/fed-isic-6client-1101`(9라운드·시드 41 탐색)과 같은 브랜치의 `results/fed_isic_6client_kaggle_1102_v2/`(확장 실험 A: 100스텝 기준 18라운드, FedBN 9라운드)에서 읽습니다.

**이번 커밋에서 돌리는 것** (GPU 2장이면 두 장에 나눠 동시에 실행)
- E1 μ 재확인(공식 설정 그대로, 검증만): 시드 41에 μ=0.03·0.3, 시드 40에 FedAvg와 μ=0.03·0.1·0.3. 계획서의 규칙(마지막 3라운드 val macro-F1 평균, 동률이면 작은 μ)으로 μ를 정해 이후 FedProx에 사용
- E4b 개선 설정 18라운드, 시드 42/100/2026, test는 `--final_test`로 1회: FedAvg·FedProx(선정 μ)·FedBN
  - `--keep_optimizer`(클라이언트 Adam 상태 유지) + `--lr_schedule cosine`(후반 학습률 감소)
  - 에폭 기준: `--num_updates 0 --local_epochs 2`(라운드당 센터가 자기 데이터를 2에폭 순회. 100스텝 고정이 아님)
  - 결과 파일 이름 꼬리표: `_keepopt_cosine_ep2`
- 분석: 확장 실험 A·기존 9라운드 결과와 같은 시드끼리 짝지어 `analysis/analyze_extended.py` 실행

**실행 전 설정(우측 Settings):** Accelerator = GPU T4 x2, Internet = On. 처음에는 `SMOKE = True`로 한 번 돌려 오류와 속도를 확인하고, 실제 커밋은 `SMOKE = False`로 `Save Version → Save & Run All (Commit)`.
결과: `/kaggle/working/results_v2b.tar.gz` (Output 탭, 체크포인트 .pt 제외).

In [ ]:
import os, subprocess, torch
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0), "| count:", torch.cuda.device_count())
!nvidia-smi
assert torch.cuda.is_available(), "GPU가 꺼져 있습니다. Settings > Accelerator에서 GPU를 선택하세요."

In [ ]:
# 1. 프로젝트 코드(확장 브랜치)와 FLamby(고정 커밋) 준비
REPO_URL = "https://github.com/NOWEAZ/fedisic-federated-framework.git"
REPO_BRANCH = "feature/v2-extended-experiments"
RESULTS_BRANCH = "results/fed-isic-6client-1101"   # 기존 6센터 3시드 결과(시드 41 탐색 + 42/100/2026 test 예측)
RESULTS_DIR = "/kaggle/working/RESULTS_OLD"
PROJECT_DIR = "/kaggle/working/CDII"
FLAMBY_DIR = "/kaggle/working/FLamby"
FLAMBY_COMMIT = "edacf54d5211520583b0133d55ac39b6fda8324b"

if not os.path.exists(PROJECT_DIR):
    !git clone -b {REPO_BRANCH} {REPO_URL} {PROJECT_DIR}
%cd {PROJECT_DIR}
!git log --oneline -1
# 확장 기능이 들어 있는 브랜치인지 확인
assert "--keep_optimizer" in open("main.py", encoding="utf-8").read() and os.path.exists("analysis/analyze_extended.py"), "확장 코드가 아닙니다"
if not os.path.exists(RESULTS_DIR):
    !git clone --depth 1 -b {RESULTS_BRANCH} {REPO_URL} {RESULTS_DIR}
REPO_OLD = os.path.join(RESULTS_DIR, "results", "fed_isic_6client_kaggle_1101")
assert os.path.exists(os.path.join(REPO_OLD, "stage1_summary.csv")), "기존 결과 폴더를 찾지 못했습니다"
!pip install -q albumentations==1.3.1
!pip install -q -r requirements.txt
if not os.path.exists(FLAMBY_DIR):
    !git clone https://github.com/owkin/FLamby.git {FLAMBY_DIR}
    !cd {FLAMBY_DIR} && git checkout {FLAMBY_COMMIT}
!cd {FLAMBY_DIR} && pip install -q -e ".[isic2019]"
import sys
if FLAMBY_DIR not in sys.path: sys.path.insert(0, FLAMBY_DIR)
import flamby; print("FLamby:", flamby.__file__)

In [ ]:
# 2. 데이터 준비. 우선순위: (a) /kaggle/input 에 붙인 전처리 데이터셋 -> (b) ISIC2019 다운로드 + FLamby 전처리
#    (ISIC 2019 / HAM10000 데이터는 CC-BY-NC 라이선스입니다.)
import glob, shutil
SAVE_V2 = "/kaggle/working/results_v2"      # E4·E3 결과 (test 1회 포함)
DATA_PATH = "/kaggle/temp/isic_data"   # 임시 디스크(출력에 포함되지 않음, 용량 여유)
for d in (SAVE_V2, DATA_PATH): os.makedirs(d, exist_ok=True)
PRE = "ISIC_2019_Training_Input_preprocessed"
PRE_DIR = os.path.join(DATA_PATH, PRE)

def have_data(): return os.path.isdir(PRE_DIR) and len(os.listdir(PRE_DIR)) > 20000

if not have_data():
    tars = glob.glob("/kaggle/input/**/*isic*preprocessed*.tar", recursive=True)
    dirs = glob.glob(f"/kaggle/input/**/{PRE}", recursive=True)
    if tars:
        print("입력 데이터셋의 tar를 풉니다:", tars[0])
        subprocess.run(["tar", "-xf", tars[0], "-C", DATA_PATH], check=True)
    elif dirs:
        print("입력 데이터셋 폴더를 복사합니다:", dirs[0])
        shutil.copytree(dirs[0], PRE_DIR)
if not have_data():
    print("전처리 데이터가 없어 ISIC2019를 다운로드·전처리합니다 (약 9GB, 수십 분).")
    SCRIPT_DIR = os.path.join(FLAMBY_DIR, "flamby", "datasets", "fed_isic2019", "dataset_creation_scripts")
    %cd {SCRIPT_DIR}
    # ISIC가 2019 파일 주소를 옮겨서 FLamby 스크립트의 옛 주소를 치환하고, 약관 질문(y/n)에는 y를 자동 전달(chang 허락 1건), 압축 해제 로그는 줄임
    !sed -i 's#isic-challenge-data.s3.amazonaws.com/2019/#isic-archive.s3.amazonaws.com/challenges/2019/#g' download_isic.py
    !sed -i 's#"unzip #"unzip -q -o #' download_isic.py
    !yes y | python download_isic.py --output-folder "{DATA_PATH}"
    !python resize_images.py
    %cd {PROJECT_DIR}
print("전처리 이미지 수:", len(os.listdir(PRE_DIR)) if os.path.isdir(PRE_DIR) else 0, "(정상: 약 23,247)")
assert have_data(), "데이터 준비 실패"

In [ ]:
# 3. 스모크 테스트: 6개 센터 로더와 학습/검증/테스트 표본 수
%cd {PROJECT_DIR}
from data_utils import get_dataloaders, BATCH_SIZE, LR, NUM_CLIENTS, NUM_EPOCHS_POOLED, get_nb_max_rounds
cl, cen, val, test = get_dataloaders(batch_size=BATCH_SIZE, num_workers=2, data_path=DATA_PATH, seed=42, val_fraction=0.1, split_seed=0)
for k in range(NUM_CLIENTS):
    print(k, len(cl[k].dataset), len(val.client_eval_loaders[k].dataset), len(test.client_eval_loaders[k].dataset))
print("BATCH", BATCH_SIZE, "LR", LR, "pooled epochs", NUM_EPOCHS_POOLED, "FL rounds", get_nb_max_rounds(100, batch_size=BATCH_SIZE))
# 기대값(train/val/test): C0 8937/993/2483, C1 2846/317/791, C2 2423/268/672, C3 1625/182/452, C4 589/66/164, C5 313/38/88
del cl, cen, val, test

In [ ]:
# 4. 공통 설정과 실행 도구 (GPU가 2장이면 작업을 두 장에 나눠 동시에 실행)
import threading, queue, re, time, json, glob, csv
from data_utils import BATCH_SIZE, LR, get_nb_max_rounds
NUM_UPDATES = 100
FL_ROUNDS = get_nb_max_rounds(NUM_UPDATES, batch_size=BATCH_SIZE)   # 공식 9 (E1은 공식 설정 그대로)
EXT_ROUNDS = 2 * FL_ROUNDS                                          # 확장 18
LOCAL_EPOCHS = 2          # 에폭 기준: 라운드당 각 센터가 자기 데이터를 2번 순회 (100스텝 고정이 아님)
MAIN_SEEDS = [42, 100, 2026]
SMOKE = False             # True면 모든 실행을 1라운드만 돌려 오류와 속도만 확인 (시드도 42 하나). 실제 커밋은 False
if SMOKE:
    MAIN_SEEDS = [42]
N_GPU = max(torch.cuda.device_count(), 1)
NUM_WORKERS = 2 if N_GPU > 1 else 4    # GPU 2장을 동시에 쓰면 CPU 코어를 나눠 씀
SAVE_MU = "/kaggle/working/results_mu" + ("_smoke" if SMOKE else "")
SAVE_V2 = "/kaggle/working/results_v2" + ("_smoke" if SMOKE else "")
for d in (SAVE_MU, SAVE_V2): os.makedirs(d, exist_ok=True)
ROUNDS_E1 = 1 if SMOKE else FL_ROUNDS
ROUNDS_MAIN = 1 if SMOKE else EXT_ROUNDS
BASE_ARGS = (f'--batch_size {BATCH_SIZE} --lr {LR} --num_workers {NUM_WORKERS} --val_fraction 0.1 --split_seed 0 '
             f'--data_path "{DATA_PATH}" --resume')

KEEP = re.compile(r"=====|\[Round|Final Test|Best Val|Traceback|Error|Resume|Model Size")
def run_job(job, gpu, log):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
    t0 = time.time()
    p = subprocess.Popen(job["cmd"], shell=True, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1, errors="replace")
    for line in p.stdout:
        line = line.strip()
        if line and KEEP.search(line):
            if "[Round" in line: line = line[line.find("[Round"):]
            print(f"[GPU{gpu} {job['name']}] {line[:240]}", flush=True)
    rc = p.wait()
    log.append((job["name"], gpu, rc, round(time.time() - t0)))
    print(f"[GPU{gpu}] 끝: {job['name']} rc={rc} {time.time()-t0:.0f}s", flush=True)

def run_jobs(jobs):
    q = queue.Queue()
    for j in jobs: q.put(j)
    log = []
    def worker(gpu):
        while True:
            try: job = q.get_nowait()
            except queue.Empty: return
            run_job(job, gpu, log)
    threads = [threading.Thread(target=worker, args=(g,)) for g in range(N_GPU)]
    for t in threads: t.start()
    for t in threads: t.join()
    bad = [l for l in log if l[2] != 0]
    print("실패한 실행:", bad if bad else "없음", flush=True)
    return log
print("GPU", N_GPU, "| workers", NUM_WORKERS, "| SMOKE", SMOKE, "| 라운드", ROUNDS_E1, ROUNDS_MAIN, "| 시드", MAIN_SEEDS)

In [ ]:
# 5. E1: μ 재확인 (공식 설정 그대로: 100스텝, 9라운드, 검증만 보고 test는 열지 않음)
# (seed, method, mu) — 시드 41의 μ=0.001·0.01·0.1·1.0 결과는 기존 결과 브랜치에서 읽는다
MU_RUNS = [(41, "fedprox", 0.03), (41, "fedprox", 0.3),
           (40, "fedavg", 0.0), (40, "fedprox", 0.03), (40, "fedprox", 0.1), (40, "fedprox", 0.3)]
mu_jobs = []
for seed, method, mu in MU_RUNS:
    mu_arg = f"--mu {mu} " if method == "fedprox" else ""
    mu_jobs.append(dict(name=f"E1 {method} mu={mu} seed={seed}",
        cmd=f'python main.py --method {method} {mu_arg}--rounds {ROUNDS_E1} --num_updates {NUM_UPDATES} --seed {seed} {BASE_ARGS} --save_dir "{SAVE_MU}"'))
log_e1 = run_jobs(mu_jobs)

In [ ]:
# 6. μ 선정: 후보 0.03/0.1/0.3 중 시드 40·41 평균 "마지막 3라운드 val macro-F1"이 가장 큰 값(동률이면 작은 μ). 실패하면 기존 0.1
def last3_f1(path):
    rows = list(csv.DictReader(open(path, encoding="utf-8")))
    return sum(float(r["macro_f1"]) for r in rows[-3:]) / 3 if len(rows) >= 3 else None
def metrics_path(method, mu, seed):
    base = (f"fedprox_mu{mu:g}" if method == "fedprox" else "fedavg") + f"_seed{seed}_metrics.csv"
    for d in (SAVE_MU, REPO_OLD):
        p = os.path.join(d, base)
        if os.path.exists(p): return p
rows_out = []
for method, mu in [("fedavg", 0.0), ("fedprox", 0.001), ("fedprox", 0.01), ("fedprox", 0.03), ("fedprox", 0.1), ("fedprox", 0.3), ("fedprox", 1.0)]:
    vals = {}
    for seed in (40, 41):
        p = metrics_path(method, mu, seed)
        vals[seed] = last3_f1(p) if p else None
    rows_out.append((method, mu, vals))
    print(f"{method:8s} mu={mu:<6g} seed40={vals[40]}  seed41={vals[41]}")
BEST_MU = 0.1
cand = []
for method, mu, vals in rows_out:
    if method == "fedprox" and mu in (0.03, 0.1, 0.3) and None not in vals.values():
        cand.append((-round((vals[40] + vals[41]) / 2, 4), mu, (vals[40] + vals[41]) / 2))
if len(cand) == 3:
    cand.sort(); BEST_MU = cand[0][1]
    print("선정된 μ:", BEST_MU, "| 후보별 평균:", {c[1]: round(c[2], 4) for c in cand})
else:
    print("후보가 모두 모이지 않아 기존 μ=0.1을 사용합니다 (SMOKE이면 정상)")
json.dump({"best_mu": BEST_MU, "table": [(m, mu, {str(k): v for k, v in vals.items()}) for m, mu, vals in rows_out]},
          open(os.path.join(SAVE_MU, "best_mu.json"), "w"), indent=1)
!cd /kaggle/working && tar -czf results_mu.tar.gz {os.path.basename(SAVE_MU)} && ls -la results_mu.tar.gz

In [ ]:
# 7. E4b: 개선 설정 18라운드 (keep_optimizer + cosine + 에폭 기준 2에폭) — FedProx(선정 μ), FedAvg, FedBN, 시드 42/100/2026
EP_ARGS = (f"--num_updates 0 --local_epochs {LOCAL_EPOCHS} --rounds {ROUNDS_MAIN} "
           f"--keep_optimizer --lr_schedule cosine --tag ep{LOCAL_EPOCHS}")
main_jobs = []
for s in MAIN_SEEDS:      # 시드 단위로 묶어 두어 중간에 끊겨도 완성된 시드가 남도록
    for method, mu_arg in (("fedprox", f"--mu {BEST_MU} "), ("fedavg", ""), ("fedbn", "")):
        main_jobs.append(dict(name=f"E4b {method} seed={s}",
            cmd=f'python main.py --method {method} {mu_arg}{EP_ARGS} --seed {s} {BASE_ARGS} --save_dir "{SAVE_V2}" --final_test'))
log_main = run_jobs(main_jobs)

In [ ]:
# 8. 분석: 같은 시드끼리 짝지어 확장 분석 (기존 9라운드 / 확장 실험 A의 100스텝 18라운드와 비교)
#    FedBN은 센터별 BN 모델로 평가한 것이라 FedAvg/FedProx(서버 모델 하나)와 평가 대상이 다르다는 점에 유의.
import shutil
AN_IN = "/kaggle/working/analysis_inputs_b"; AN_OUT = "/kaggle/working/analysis_extended_b"
os.makedirs(AN_IN, exist_ok=True)
A_DIR = os.path.join(PROJECT_DIR, "results", "fed_isic_6client_kaggle_1102_v2")   # 확장 실험 A 결과
NEW = {"avg": f"fedavg_keepopt_cosine_ep{LOCAL_EPOCHS}", "prox": f"fedprox_mu{BEST_MU:g}_keepopt_cosine_ep{LOCAL_EPOCHS}",
       "bn": f"fedbn_keepopt_cosine_ep{LOCAL_EPOCHS}"}
def cp(d, pat, s):
    p = os.path.join(d, f"{pat}_seed{s}_final_test_preds.npz")
    if os.path.exists(p): shutil.copy(p, AN_IN)
    else: print("없음:", p)
for s in MAIN_SEEDS:
    for pat in ("centralized", "fedavg", "fedprox_mu0.1"): cp(REPO_OLD, pat, s)
    for pat in ("fedavg_r18", "fedprox_mu0.1_r18", "fedbn"): cp(A_DIR, pat, s)
    for pat in NEW.values(): cp(SAVE_V2, pat, s)
pairs = [f"{NEW['avg']}:fedavg_r18", f"{NEW['prox']}:fedprox_mu0.1_r18", f"{NEW['prox']}:{NEW['avg']}",
         f"{NEW['bn']}:{NEW['avg']}", f"{NEW['bn']}:fedbn", f"{NEW['avg']}:fedavg", f"{NEW['prox']}:fedprox_mu0.1"]
!python analysis/analyze_extended.py --results_dir "{AN_IN}" --out_dir "{AN_OUT}" --pairs {" ".join(pairs)}
import pandas as pd
display(pd.read_csv(os.path.join(AN_OUT, "paired_summary.csv")))
display(pd.read_csv(os.path.join(AN_OUT, "bootstrap_ci_overall.csv")))

In [ ]:
# 9. 요약과 묶기 (체크포인트 .pt는 제외)
rows = []
for p in sorted(glob.glob(os.path.join(SAVE_V2, "*_final_test.json"))):
    d = json.load(open(p))["final"]
    rows.append({"file": os.path.basename(p), **{k: d[k] for k in ("balanced_acc", "macro_f1", "ece", "bal_acc_gap", "bal_acc_std")}})
fb = pd.DataFrame(rows); display(fb)
fb["method"] = fb["file"].str.replace(r"_seed\d+_final_test\.json", "", regex=True)
display(fb.drop(columns="file").groupby("method").agg(["mean", "std"]))
!cd /kaggle/working && tar --exclude='*.pt' -czf results_v2b.tar.gz {os.path.basename(SAVE_MU)} {os.path.basename(SAVE_V2)} analysis_extended_b && ls -la results_v2b.tar.gz